In [ ]:
import pandas as pd
import numpy as np
import gzip
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import warnings
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score,roc_curve,auc
from scipy.stats import chi, chi2
import statsmodels.api as sm
import toad
warnings.filterwarnings('ignore')

In [ ]:
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号,#有中文出现的情况，需要u'内容'
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.max_row', 1000)
pd.options.display.max_info_columns = 200

In [ ]:
#data = pd.read_csv("Loan_status_2007-2020Q3.gzip",index_col=0) ### data from lendingClub/Kaggle

# 对样本和变量进行筛选

In [ ]:
#限定样本小微企业
#data1 = data[data['purpose'] == 'small_business']

In [ ]:
#data1.to_csv('loan_status.csv')

In [ ]:
data1 = pd.read_csv('loan_status.csv',index_col=0)

In [ ]:
data1.info()

In [ ]:
#剔除不需要的变量
useless_fea = ['id','next_pymnt_d', 'issue_d','earliest_cr_line','last_credit_pull_d','last_pymnt_d','url','zip_code',
 'addr_state','sub_grade','title']

In [ ]:
for i in useless_fea:
    data1.drop(columns = i, inplace=True)

In [ ]:
data1

In [ ]:
#剔除模型唯一值的变量
var_same=[]
for i in data1.columns:
    mode_value = data1[i].mode()[0]
    mode_rate = len(data1[i][data1[i]==mode_value]) / data1.shape[0]
    if mode_rate > 0.9:
        var_same.append(i)
        data1.drop(i,axis=1,inplace=True)

In [ ]:
print(f'There are {data1.isnull().any().sum()} columns in train dataset with missing values.')

In [ ]:
#剔除缺失值大于50%的变量
have_null_fea_dict = (data1.isnull().sum()/len(data1)).to_dict()
fea_null_moreThanHalf = {}
for key,value in have_null_fea_dict.items():
    if value > 0.5:
        fea_null_moreThanHalf[key] = value

In [ ]:
Alist = list(fea_null_moreThanHalf.keys())

In [ ]:
for i in Alist:
    data1.drop(columns = i,inplace=True)

In [ ]:
data1

In [ ]:
#变量筛选完毕
data1.to_csv('loan_status2.csv')

# 变量转换和缺失值填充

In [ ]:
data1 = pd.read_csv('loan_status2.csv',index_col=0)

In [ ]:
data1.info()

In [ ]:
#打印数值特征和类别特征
numerical_fea = list(data1.select_dtypes(exclude=['object']).columns)
category_fea = list(filter(lambda x: x not in numerical_fea,list(data1.columns)))

In [ ]:
data1['loan_status'].unique()

In [ ]:
#对非数值型变量进行转换
def change(x):
    if x=='Fully Paid':
        return 0
    elif x=='Current':
        return 0
    elif x=='Does not meet the credit policy. Status:Fully Paid':
        return 0
    elif x=='In Grace Period':
        return 0
    elif x=='Charged Off':
        return 1
    elif x=='Does not meet the credit policy. Status:Charged Off':
        return 1
    elif x=='Late (16-30 days)':
        return 1
    elif x=='Late (31-120 days)':
        return 1
    elif x=='Default':
        return 1
    else:
        return 1

data1['loan_status'] = data1['loan_status'].apply(lambda x:change(x))

In [ ]:
data1['loan_status'].unique()

In [ ]:
data1['emp_length'].value_counts(dropna=False).sort_index()

In [ ]:
data1['emp_length'] = data1['emp_length'].map({'0':0,'< 1 year':1,'1 year':2,'2 years':3,'3 years':4,'4 years':5,'5 years':6,
        '6 years':7,'7 years':8,'8 years':9,'9 years':10,'10+ years':11})

In [ ]:
data1['emp_length']

In [ ]:
data1['term'] = data1['term'].apply(lambda s: int(s[0:3]))

In [ ]:
data1['term']

In [ ]:
data1['int_rate'] = data1['int_rate'].str.replace("%","").astype("float")

In [ ]:
data1["revol_util"] = data1["revol_util"].str.replace("%","").astype("float")

In [ ]:
data1['grade'] = data1['grade'].map({'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6})

In [ ]:
data1['initial_list_status'] = data1['initial_list_status'].map({"w":0, "f":1})

In [ ]:
data1['verification_status'] = data1['verification_status'].map({'Not Verified':0,'Source Verified':1,'Verified':2})

In [ ]:
data1['home_ownership'] = data1['home_ownership'].map({'ANY':0,'RENT':1,'OWN':2,'MORTGAGE':3, "OTHER": 4})

In [ ]:
emp_title_dict = {}
idx = 0
for item in data1.emp_title:
    if not pd.isnull(item) and item not in emp_title_dict:
        emp_title_dict[item] = idx
        idx += 1

In [ ]:
data1['emp_title'] = data1['emp_title'].map(emp_title_dict)

In [ ]:
#缺失值处理

In [ ]:
features1 = ['acc_open_past_24mths','avg_cur_bal','bc_open_to_buy','bc_util','dti','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op'
            ,'mo_sin_rcnt_tl','mort_acc','mths_since_recent_bc','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl'
            ,'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats','num_tl_90g_dpd_24m'
            ,'num_tl_op_past_12m','pct_tl_nvr_dlq','percent_bc_gt_75','pub_rec_bankruptcies','tot_coll_amt','tot_cur_bal','tot_hi_cred_lim'
            ,'total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit','total_rev_hi_lim']

In [ ]:
features2 = ['all_util','emp_title','emp_length','il_util','inq_fi','inq_last_12m','max_bal_bc','mo_sin_old_il_acct'
            ,'mths_since_last_delinq','mths_since_rcnt_il','mths_since_recent_inq','open_acc_6m','open_act_il','open_il_12m'
            ,'open_il_24m','open_rv_12m','open_rv_24m','revol_util','total_bal_il','total_cu_tl']

In [ ]:
for i in features1:
    median_value = data1[i].median()
    data1[i] = data1[i].fillna(median_value)

In [ ]:
for i in features2:
    data1[i] = data1[i].fillna(-99)

In [ ]:
mode_value = data1['num_tl_120dpd_2m'].mode()[0]
data1['num_tl_120dpd_2m'] = data1['num_tl_120dpd_2m'].fillna(mode_value)

In [ ]:
data1.reset_index(drop=True,inplace=True)

In [ ]:
#导出文件
data1.to_csv('loan_status3.csv')

# 变量分箱和特征筛选

In [ ]:
#读取文件
data2 = pd.read_csv('loan_status3.csv',index_col=0)

In [ ]:
Xtr,Xts,Ytr,Yts = train_test_split(data2.drop('loan_status',axis=1),data2['loan_status'],test_size=0.3,random_state=450)

data_tr = pd.concat([Xtr,Ytr],axis=1)

data_ts = pd.concat([Xts,Yts],axis=1)
print(data_tr.shape)

In [ ]:
data_tr['loan_status'].value_counts()

In [ ]:
#查看EDA报告
toad.detector.detect(data_tr)

In [ ]:
#根据iv和corr筛选特征变量
selected_train, drop_lst = toad.selection.select(
                         data_tr,target = 'loan_status', iv=0.02, corr = 0.75, return_drop=True)
selected_test = data_ts[selected_train.columns]
print(drop_lst)

In [ ]:
quality = toad.quality(selected_train,'loan_status',iv_only=True)
quality.sort_values('iv',ascending=False)
#剩余39个变量

In [ ]:
#进行分箱和woe编码

In [ ]:
combiner = toad.transform.Combiner()

In [ ]:
combiner.fit(selected_train,y='loan_status',method='chi',min_samples = 0.05)

In [ ]:
bins = combiner.export()
bins

In [ ]:
binned_data = combiner.transform(selected_train)

In [ ]:
transer = toad.transform.WOETransformer()
data_tr_woe = transer.fit_transform(binned_data, binned_data['loan_status'], exclude=['loan_status'])
data_ts_woe = transer.transform(combiner.transform(selected_test))

In [ ]:
data_tr_woe

In [ ]:
#逐步回归变量选择
final_train, droplist2 = toad.selection.stepwise(data_tr_woe, target='loan_status', direction = 'both', criterion = 'aic',return_drop=True)
final_test = data_ts_woe[final_train.columns]
print(droplist2)
print(final_train.shape)
print(final_train.columns)
#剩余20个变量入模

In [ ]:
final_train

In [ ]:
###

###corr = final_train.corr() 
###fig, ax = plt.subplots(figsize=(16, 16))
###sns.heatmap(corr, mask=np.zeros_like(corr, dtype=np.bool), cmap=sns.diverging_palette(220, 10, as_cmap=True), square=True, ax=ax)
###plt.show()

###

In [ ]:
#最后再对入模变量进行一次筛选，根据不同变量属性整体的iv值

In [ ]:
delete_fea = ['initial_list_status','mo_sin_old_rev_tl_op','mths_since_recent_bc','total_bal_ex_mort'
              ,'num_actv_rev_tl','recoveries','grade','emp_title','verification_status']

In [ ]:
for i in delete_fea:
    final_train.drop(columns = i, inplace=True)

In [ ]:
final_test = final_test[final_train.columns]

In [ ]:
#建模

In [ ]:
Xtr = final_train.drop(['loan_status'],axis=1)
Ytr = final_train['loan_status']
Xts = final_test.drop(['loan_status'],axis=1)
Yts = final_test['loan_status']

In [ ]:
Xtr1=sm.add_constant(Xtr)
logit=sm.Logit(Ytr,Xtr1).fit()
logit.summary()

In [ ]:
model = LogisticRegression()
lr = model.fit(Xtr,Ytr)

EYtr_proba = lr.predict_proba(Xtr)[:,1]
EYtr = lr.predict(Xtr)

print('Training error')
print('KS:', KS(EYtr_proba,Ytr))
print('AUC:', AUC(EYtr_proba,Ytr))

EYts_proba = lr.predict_proba(Xts)[:,1]
EYts = lr.predict(Xts)

print('\nTest error')
print('KS:', KS(EYts_proba,Yts))
print('AUC:', AUC(EYts_proba,Yts))

In [ ]:
### plot ROC curve
fpr_train,tpr_train,thred_train = roc_curve(Ytr,EYtr_proba)
fpr_test,tpr_test,thred_test  = roc_curve(Yts,EYts_proba)

label = ['Train - AUC:{:.4f}'.format(auc(fpr_train,tpr_train)),
             'Test - AUC:{:.4f}'.format(auc(fpr_test,tpr_test))]
plt.plot(fpr_train,tpr_train)
plt.plot(fpr_test,tpr_test)
plt.plot([0,1],[0,1],'d--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(label, loc = 4)
plt.title('ROC Curve')


In [ ]:
### plot test ROC curve
plt.plot(fpr_test,tpr_test)
plt.plot([0,1],[0,1],'d--')
plt.title('ROC Curve')
label = ['Test - AUC:{:.4f}'.format(auc(fpr_test,tpr_test))]
plt.legend(label, loc = 4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.show()

In [ ]:
### plot test KS Curve
plt.plot(thred_test[1:],tpr_test[1:])
plt.plot(thred_test[1:],fpr_test[1:])
plt.plot(thred_test[1:],tpr_test[1:]-fpr_test[1:])
plt.title('KS Curve')
plt.legend(['tpr','fpr','tpr-fpr'])
plt.xlabel('threshold')
plt.gca().invert_xaxis()
plt.show()

In [ ]:
tr_bucket = toad.metrics.KS_bucket(EYtr_proba,Ytr,bucket=10,method='quantile')
tr_bucket

In [ ]:
card = toad.ScoreCard(combiner = combiner, transer = transer , C = 0.1, class_weight = 'balanced',
    base_score = 600,
    base_odds = 1/100 ,
    pdo = 50,
    rate = 2)
card.fit(Xtr, Ytr)
card1 = card.export(to_frame = True)
print(card1)

In [ ]:
def dict_type(card1):
    card1.score=card1.score.round()
    card1.value.fillna('nan',inplace=True)
    namelist=list(set(card1.name))
    myvalue=[]
    for var in namelist:
        ind_loc=card1.name==var
        value_dict = dict(zip(card1.value.loc[ind_loc],card1.score.loc[ind_loc]))
        myvalue=myvalue+[value_dict]
    big_dict = dict(zip(namelist,myvalue))
    return big_dict
    
# card2是四舍五入的新卡
card2 = dict_type(card1)

# 重新拟合card
card = toad.ScoreCard(combiner = combiner, transer = transer , card = card2,C = 0.1, class_weight = 'balanced',
    base_score = 600,
    base_odds = 1/100 ,
    pdo = 50,
    rate = 2)  
# 将评分卡结果导出并保存
final_card = card.export(to_frame=True,) 
final_card

In [ ]:
#得到样本的最终评分
final_score=pd.DataFrame(card.predict(data_ts),index=data_ts.index,columns=['score'])
final_score.describe()

In [ ]:
data_score = pd.concat([final_test,final_score],axis = 1)

In [ ]:
data_score

In [ ]:
def cal_df(data, cut_num, feature, target):
    # 1、数据分箱（等宽）
    data_cut = pd.cut(data[feature], cut_num)
    # 2、统计各个分箱的总样本数、违约样本数和不违约样本数
    cut_group_all = data[target].groupby(data_cut).count()#总样本数
    cut_y = data[target].groupby(data_cut).sum() #违约样本数
    cut_n = cut_group_all - cut_y #未违约样本数
    # 3、汇总基础数据
    df = pd.DataFrame() #创建空DataFrame用来汇总数据
    df['总数'] = cut_group_all
    df['违约'] = cut_y
    df['未违约'] = cut_n
    # 4、统计违约数比率
    
    df['违约率%'] = df['违约'] / df['总数']
    df['违约率%'] = df['违约率%'].apply(lambda x: format(x, '.2%'))
   
    return df

In [ ]:
#训练集
score_card = cal_df(data_score, 10, 'score', 'loan_status')
score_card